In [1]:
import numpy as np
import pandas as pd

In [4]:
import os

absolute_path = os.getcwd()
print('ROUTE', absolute_path)

ROUTE C:\Users\lidia\Documents\UNIR - máster\Python\ejercicio3_analisis_ventas


In [8]:
sales = pd.read_csv(f'{absolute_path}/data/sales.csv')
inventories = pd.read_csv(f'{absolute_path}/data/inventories.csv')
satisfaction = pd.read_csv(f'{absolute_path}/data/satisfaction.csv')

df_sales = pd.DataFrame(sales)
df_inventories = pd.DataFrame(inventories)
df_satisfaction = pd.DataFrame(satisfaction)

In [10]:
# Limpieza de datos (eliminación de NaN)
df_sales = df_sales.dropna()
df_inventories = df_inventories.dropna()
df_satisfaction = df_satisfaction.dropna()

In [11]:
# Número total de valores nulos
print("Valores nulos en sales:", sales.isna().sum().sum())
print("Valores nulos en inventories:", inventories.isna().sum().sum())
print("Valores nulos en satisfaction:", satisfaction.isna().sum().sum())

Valores nulos en sales: 0
Valores nulos en inventories: 0
Valores nulos en satisfaction: 0


## Exploración de datos

In [15]:
df_sales.dtypes

ID_Tienda           int64
Producto              str
Cantidad_Vendida    int64
Precio_Unitario     int64
Fecha_Venta           str
dtype: object

In [16]:
# Ventas totales por producto y tienda
df_sales.groupby(['Producto', 'ID_Tienda'])['Cantidad_Vendida'].sum()

Producto    ID_Tienda
Producto A  1            20
            2            30
            3            10
            4            25
Producto B  1            15
            3            40
            5            20
Producto C  2            25
            4            35
            5            30
Name: Cantidad_Vendida, dtype: int64

In [17]:
# Ingresos totales por tienda
df_sales['Importe_Total'] = df_sales['Cantidad_Vendida'] * df_sales['Precio_Unitario']
df_sales.groupby('ID_Tienda')['Importe_Total'].sum()

ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Importe_Total, dtype: int64

In [18]:
# Resumen estadístico de ventas
df_sales.describe()

,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Importe_Total
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000


In [32]:
# Promedio de ventas por tienda
df_sales.groupby('ID_Tienda')['Importe_Total'].mean()
# Promedio de ventas por tienda y categoría de producto
df_sales.groupby(['ID_Tienda', 'Producto'])['Importe_Total'].mean()

ID_Tienda  Producto  
1          Producto A     2000.0
           Producto B     3000.0
2          Producto A     3000.0
           Producto C     7500.0
3          Producto A     1000.0
           Producto B     8000.0
4          Producto A     2500.0
           Producto C    10500.0
5          Producto B     4000.0
           Producto C     9000.0
Name: Importe_Total, dtype: float64

## Análisis de inventarios

In [19]:
df_inventories.dtypes

ID_Tienda              int64
Producto                 str
Stock_Disponible       int64
Fecha_Actualización      str
dtype: object

In [27]:
# Cálculo de rotación de inventario por tienda
# (división ventas totales por stock disponible por producto)
ventas_tienda = df_sales.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum()
rotacion_inventario = df_inventories.merge(ventas_tienda, on=['ID_Tienda', 'Producto']).rename(columns={'Cantidad_Vendida': 'Ventas_Tienda'})
rotacion_inventario['Rotacion'] = rotacion_inventario['Ventas_Tienda'] / rotacion_inventario['Stock_Disponible']
rotacion_inventario = rotacion_inventario.drop(columns='Ventas_Tienda')
print(rotacion_inventario)

   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  Rotacion
0          1  Producto A                50          2023-01-05  0.400000
1          1  Producto B                40          2023-01-06  0.375000
2          2  Producto A                60          2023-01-07  0.500000
3          2  Producto C                45          2023-01-08  0.555556
4          3  Producto A                30          2023-01-09  0.333333
5          3  Producto B                80          2023-01-10  0.500000
6          4  Producto C                70          2023-01-11  0.500000
7          4  Producto A                50          2023-01-12  0.500000
8          5  Producto B                40          2023-01-13  0.500000
9          5  Producto C                60          2023-01-14  0.500000


In [28]:
# Almacena resultado en nueva columna de DataFrame Inventario
df_inventories = df_inventories.merge(rotacion_inventario[['ID_Tienda', 'Producto', 'Rotacion']], on=['ID_Tienda', 'Producto'])
print(df_inventories)

   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  Rotacion
0          1  Producto A                50          2023-01-05  0.400000
1          1  Producto B                40          2023-01-06  0.375000
2          2  Producto A                60          2023-01-07  0.500000
3          2  Producto C                45          2023-01-08  0.555556
4          3  Producto A                30          2023-01-09  0.333333
5          3  Producto B                80          2023-01-10  0.500000
6          4  Producto C                70          2023-01-11  0.500000
7          4  Producto A                50          2023-01-12  0.500000
8          5  Producto B                40          2023-01-13  0.500000
9          5  Producto C                60          2023-01-14  0.500000


In [39]:
# Tiendas con niveles críticos de inventario. Aquellas que tengan % productos vendidos < 10% del stock disponible
ventas_stock = df_sales[['ID_Tienda', 'Producto', 'Cantidad_Vendida']].merge(df_inventories[['ID_Tienda', 'Producto', 'Stock_Disponible']], on=['ID_Tienda', 'Producto'])
ventas_stock['Porcentaje_Vendido'] = ventas_stock['Cantidad_Vendida'] / ventas_stock['Stock_Disponible'] * 100
stock_limit = 10
tiendas_criticas = ventas_stock.query('Porcentaje_Vendido < @stock_limit')[['ID_Tienda', 'Producto', 'Porcentaje_Vendido']]
print(tiendas_criticas)

# Agrupar productos críticos por tienda
resumen_tiendas_criticas = (
    tiendas_criticas
    .groupby('ID_Tienda')
    .agg(
        Numero_Productos_Criticos=('Producto', 'count'),
        Productos_Criticos=('Producto', lambda x: ', '.join(x))
    )
    .reset_index()
)

print(resumen_tiendas_criticas)

Empty DataFrame
Columns: [ID_Tienda, Producto, Porcentaje_Vendido]
Index: []
Empty DataFrame
Columns: [ID_Tienda, Numero_Productos_Criticos, Productos_Criticos]
Index: []


## Satisfacción del cliente

In [40]:
# Análisis de la satisfacción de los clientes en cada tienda
# Relación con el rendimiento de las ventas
rendimiento_ventas = (df_sales.groupby('ID_Tienda')['Importe_Total']
                      .agg(['sum', 'mean'])
                      .reset_index()
                      .rename(columns={
                            'sum': 'Ventas_Totales',
                            'mean': 'Ventas_Promedio'
                        })
)
relacion_satisfaccion_rendimiento = pd.merge(
    df_satisfaction[['ID_Tienda', 'Satisfacción_Promedio']],
    rendimiento_ventas,
    on='ID_Tienda',
    how='inner'
)
print(relacion_satisfaccion_rendimiento)

   ID_Tienda  Satisfacción_Promedio  Ventas_Totales  Ventas_Promedio
0          1                     85            5000           2500.0
1          2                     90           10500           5250.0
2          3                     70            9000           4500.0
3          4                     65           13000           6500.0
4          5                     55           13000           6500.0


Se puede ver no hay una relación clara entre la satisfacción del cliente en cada tienda y la cantidad de ventas de la misma.

In [41]:
# Filtrado de tiendas con niveles bajos de satisfacción (<60%)
satisfaccion_limit = 60
tiendas_baja_satisfaccion = df_satisfaction.query('Satisfacción_Promedio < @satisfaccion_limit')
print(tiendas_baja_satisfaccion)

   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación
4          5                     55       2023-01-15


### Recomendaciones:
Se recomienda realize un estudio más exhaustivo en esta tienda para ver por qué los clientes no salen tan satisfechos como en las otras sucursales.
Se trata de falta de personal o de trato hacia los clientes?

## Operaciones con Numpy

In [44]:
# Cálculo mediana y desviación ventas totales
ventas_totales = df_sales['Importe_Total'].to_numpy()
print(ventas_totales)
mediana = np.median(ventas_totales)
print('Mediana:', mediana)
std = np.std(ventas_totales)
print('Desviación estándar:', std)
media = np.mean(ventas_totales)
print('Media:', media)

[ 2000  3000  3000  7500  1000  8000 10500  2500  4000  9000]
Mediana: 3500.0
Desviación estándar: 3189.4356867634124
Media: 5050.0


## Estimación de ventas

Para la estimación de ventas se utiliza una distribución de Poisson para simular las cantidades vendidas, dado que se trata de valores enteros no negativos. Para los precios unitarios se utiliza una distribución normal, limitando los resultados negativos a cero. Finalmente, se calculan los importes totales multiplicando las cantidades estimadas por los precios unitarios.

In [88]:
# Estimación de ventas
rng = np.random.seed(42)
narray = 90
precios_unitarios = df_sales['Precio_Unitario'].to_numpy()
estimacion_precios_unitarios = np.random.normal(precios_unitarios.mean(), precios_unitarios.std(), narray)
estimacion_precios_unitarios = np.maximum(estimacion_precios_unitarios, 0)
cantidades = df_sales['Cantidad_Vendida'].to_numpy()
estimacion_cantidades = np.random.poisson(cantidades.mean(), narray)
estimacion_ventas_totales = estimacion_precios_unitarios * estimacion_cantidades
tiendas = np.random.choice(range(1, 6), narray)
productos = np.random.choice(['Producto A', 'Producto B', 'Producto C'], narray)
estimacion_ventas = pd.DataFrame({
    'ID_Tienda': tiendas,
    'Producto': productos,
    'Cantidad_Vendida': np.floor(estimacion_cantidades),
    'Precio_Unitario': np.round(estimacion_precios_unitarios, 2),
    'Importe_Total': np.round(estimacion_ventas_totales, 2)
})
print(estimacion_ventas)

    ID_Tienda    Producto  Cantidad_Vendida  Precio_Unitario  Importe_Total
0           1  Producto C                32           231.26        7400.33
1           5  Producto A                22           178.51        3927.33
2           2  Producto A                21           243.80        5119.82
3           2  Producto A                24           316.51        7596.30
4           5  Producto C                18           170.55        3069.90
..        ...         ...               ...              ...            ...
85          1  Producto C                23           148.32        3411.38
86          4  Producto A                13           266.04        3458.51
87          1  Producto C                31           217.31        6736.55
88          5  Producto A                25           145.99        3649.87
89          3  Producto C                25           232.64        5815.88

[90 rows x 5 columns]


### Estadísticas básicas sobre las proyecciones obtenidas

In [90]:
estimacion_ventas.describe()

,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Importe_Total
count,90.000000,90.000000,90.000000,90.000000
mean,2.933333,25.266667,182.098889,4585.451111
std,1.467245,5.284703,76.876633,2221.182508
min,1.000000,10.000000,0.000000,0.000000
25%,2.000000,22.000000,140.042500,3043.522500
50%,3.000000,25.500000,179.450000,4403.990000
75%,4.000000,28.000000,232.295000,5920.992500
max,5.000000,38.000000,343.860000,11003.580000


In [91]:
# 1. Media de ventas por tienda y producto
estimacion_ventas.groupby(['ID_Tienda', 'Producto'])['Importe_Total'].mean()

ID_Tienda  Producto  
1          Producto A    4536.257143
           Producto B    4498.132000
           Producto C    4723.616250
2          Producto A    5512.006000
           Producto B    3491.540000
           Producto C    3971.156000
3          Producto A    3290.810000
           Producto B    5062.945000
           Producto C    5046.476000
4          Producto A    5010.818333
           Producto B    5373.987500
           Producto C    5155.640000
5          Producto A    4254.456923
           Producto B    5390.550000
           Producto C    3212.023333
Name: Importe_Total, dtype: float64

In [92]:
# 2. Total de productos vendidos por Producto
estimacion_ventas.groupby('Producto')['Cantidad_Vendida'].sum()

Producto
Producto A    803
Producto B    662
Producto C    809
Name: Cantidad_Vendida, dtype: int32

In [93]:
# 3. Media de ventas por tienda
estimacion_ventas.groupby('ID_Tienda')['Cantidad_Vendida'].mean()

ID_Tienda
1    26.000000
2    23.400000
3    25.733333
4    27.000000
5    24.631579
Name: Cantidad_Vendida, dtype: float64